In [1]:
"""
Data Engineering and Processing
"""

import json
import pandas as pd
import numpy as np

RAW_PATH = "meteo-historical-data.json"
OUT_CSV_PATH = "Processed Data.csv"

pd.set_option("display.width", 145)
pd.set_option("display.max_columns", 55)

print("=" * 60)
print("STEP 1: Loading Data")
print("=" * 60)

with open(RAW_PATH, "r") as f:
    raw = json.load(f)

df = pd.DataFrame(raw["observation_grided_data"])
print(f"Raw shape: {df.shape}")
print(f"Columns  : {list(df.columns)}")

rename_map = {
    "observe_grid10_id": "Grid_id",
    "observe_grid10_lat": "Latitude",
    "observe_grid10_lon": "Longitude",
    "observe_grid10_year": "Year",
    "observe_grid10_month": "Month",
    "observe_grid10_tas": "tas",
    "observe_grid10_tasmin": "tasmin",
    "observe_grid10_tasmax": "tasmax",
    "observe_grid10_pr": "pr",
    "observe_grid10_district": "District",
}
df = df.rename(columns=rename_map)

df["grid_cell_id"] = (
    df["Latitude"].astype(str) + "_" + df["Longitude"].astype(str)
)

print(f"\nDistricts found ({df['District'].nunique()}): "
      f"{sorted(df['District'].unique())}")
print(f"Unique grid cells: {df['grid_cell_id'].nunique()}")
print(f"Year range: {df['Year'].min()} - {df['Year'].max()}")

print("\n" + "=" * 60)
print("STEP 2: Checking Duplicates")
print("=" * 60)

full_row_dupes = df.duplicated().sum()
print(f"Duplicated rows: {full_row_dupes}")

logical_dupes = df.duplicated(
    subset=["grid_cell_id", "Year", "Month"]
).sum()
print(f"Duplicate (grid cell, year, month) combinations: {logical_dupes}")

if full_row_dupes > 0:
    df = df.drop_duplicates()
    print("-> Duplicated rows removed.")

if logical_dupes > 0:
    df = df.drop_duplicates(subset=["grid_cell_id", "Year", "Month"],
                             keep="first")
    print("-> Duplicates removed (kept first occurrence).")

if full_row_dupes == 0 and logical_dupes == 0:
    print("-> No duplicates in the original dataset. Data is clean.")

print("\n" + "=" * 60)
print("STEP 3: Check for Missing Values in the Dataset")
print("=" * 60)

missing = df.isnull().sum()
print(missing)

if missing.sum() == 0:
    print("-> No missing values were found in the Original dataset.")
else:
    num_cols = ["tas", "tasmin", "tasmax", "pr"]
    for col in num_cols:
        if df[col].isnull().sum() > 0:
            df[col] = df.groupby("Grid_id")[col].transform(
                lambda s: s.fillna(s.median())
            )
    print("-> Missing numeric values were imputed using median.")
    print(df.isnull().sum())

df = df.sort_values(["grid_cell_id", "Year", "Month"]).reset_index(drop=True)

expected_months_per_cell = df["Year"].nunique() * 12
actual_counts = df.groupby("grid_cell_id").size()
gaps = actual_counts[actual_counts != expected_months_per_cell]
print(f"\nExpected records per grid cell (if no time gaps): "
      f"{expected_months_per_cell}")
if len(gaps) == 0:
    print("-> There's a complete gap-free monthly time series  on every grid cell.")
else:
    print(f"-> {len(gaps)} grid cell(s) have incomplete time series:")
    print(gaps)

print("\n" + "=" * 60)
print("STEP 4: Season Denotation")
print("=" * 60)


def assign_season(month: int) -> str:
    if month in (9, 10, 11, 12):
        return "Season A (Rainy)"
    elif month in (3, 4, 5):
        return "Season B (Rainy)"
    else:
        return "Dry Season"


df["Season"] = df["Month"].apply(assign_season)
print(df["Season"].value_counts())

print("\n" + "=" * 60)
print("STEP 5: Fature Engineering")
print("=" * 60)

df["temp_range"] = df["tasmax"] - df["tasmin"]
print("-> Created 'temp_range' = tasmax - tasmin")

for lag in (1, 2, 3):
    df[f"pr_lag{lag}"] = df.groupby("grid_cell_id")["pr"].shift(lag)
print("-> Created 'pr_lag1', 'pr_lag2', 'pr_lag3'")

lag_cols = ["pr_lag1", "pr_lag2", "pr_lag3"]
print(f"\nRows with at least one NaN lag (start of each grid cell series): "
      f"{df[lag_cols].isnull().any(axis=1).sum()}")

for col in lag_cols:
    df[col] = df.groupby("grid_cell_id")[col].transform(
        lambda s: s.fillna(s.mean())
    )
print("-> Edge-of-series NaN lags filled with each grid cell's mean rainfall.")

district_p95 = df.groupby("District")["pr"].quantile(0.95)
df["district_pr_p95"] = df["District"].map(district_p95)
df["is_extreme_rainfall"] = (
    df["pr"] > df["district_pr_p95"]
).astype(int)
print("-> Created 'district_pr_p95' (95th percentile rainfall per district climatology)")
print("-> Created 'is_extreme_rainfall' (1 = extreme rainfall event)")
print(df["is_extreme_rainfall"].value_counts(normalize=True).rename("proportion"))

df["date"] = pd.to_datetime(
    df["Year"].astype(str) + "-" + df["Month"].astype(str) + "-01"
)

print("\n" + "=" * 60)
print("STEP 6: One Hot Encoding Districts")
print("=" * 60)

district_dummies = pd.get_dummies(
    df["District"], prefix="District", dtype=int
)
print(f"One-hot columns created: {list(district_dummies.columns)}")

df = pd.concat([df, district_dummies], axis=1)

print("\n" + "=" * 60)
print("STEP 7: Saving")
print("=" * 60)

ordered_cols = (
    ["Grid_id", "Latitude", "Longitude", "District"]
    + list(district_dummies.columns)
    + ["Year", "Month", "date", "Season"]
    + ["tas", "tasmin", "tasmax", "temp_range"]
    + ["pr", "pr_lag1", "pr_lag2", "pr_lag3"]
    + ["district_pr_p95", "is_extreme_rainfall"]
)
df = df[ordered_cols]

print(f"Final processed shape: {df.shape}")
print(f"Final columns ({len(df.columns)}): {list(df.columns)}")
print("\nSample rows:")
print(df.head(5))

df.to_csv(OUT_CSV_PATH, index=False)
print(f"\nProcessed data saved to: {OUT_CSV_PATH}")

print("\n" + "=" * 60)
print("Data Preprocessing successfully complete")
print("=" * 60)


STEP 1: Loading Data
Raw shape: (18720, 10)
Columns  : ['observe_grid10_id', 'observe_grid10_lat', 'observe_grid10_lon', 'observe_grid10_year', 'observe_grid10_month', 'observe_grid10_tas', 'observe_grid10_tasmin', 'observe_grid10_tasmax', 'observe_grid10_pr', 'observe_grid10_district']

Districts found (7): ['Burera', 'Gakenke', 'Musanze', 'Ngororero', 'Nyabihu', 'Rubavu', 'Rutsiro']
Unique grid cells: 40
Year range: 1983 - 2021

STEP 2: Checking Duplicates
Duplicated rows: 0
Duplicate (grid cell, year, month) combinations: 0
-> No duplicates in the original dataset. Data is clean.

STEP 3: Check for Missing Values in the Dataset
Grid_id         0
Latitude        0
Longitude       0
Year            0
Month           0
tas             0
tasmin          0
tasmax          0
pr              0
District        0
grid_cell_id    0
dtype: int64
-> No missing values were found in the Original dataset.

Expected records per grid cell (if no time gaps): 468
-> There's a complete gap-free monthly